In [4]:
import pandas as pd
import json

with open("/p/llmreliability/test_repos/tree-of-thought-llm/logs/gsm8k/cleaned_claude_responses.json", "r") as file:
    data = json.load(file)
    
df = pd.DataFrame(data)

# Convert model_response and correct_answer to the same type (string)
df['extracted_answer'] = df['extracted_answer'].astype(str)
df['correct_answer'] = df['correct_answer'].astype(str)


In [5]:
# Calculate accuracy
df['is_correct'] = df['extracted_answer'] == df['correct_answer']
accuracy = df['is_correct'].mean()

In [6]:
print(f"Total questions: {len(df)}")
print(f"Correct answers: {df['is_correct'].sum()}")
print(f"Accuracy: {accuracy:.2%}")

Total questions: 150
Correct answers: 0
Accuracy: 0.00%


In [5]:
import pandas as pd
import json
import re

# Load the data
with open('/p/llmreliability/test_repos/tree-of-thought-llm/logs/gsm8k/gpt-4o_0.7_propose1_value1_greedy1_start0_end100.json', 'r') as f:
    data = json.load(f)

# Create a DataFrame
df = pd.DataFrame(data)

# Function to extract the final answer from the full response
def extract_final_answer(full_response):
    # Look for the answer after '####' or at the end of the string
    match = re.search(r'####\s*(\d+)', full_response)
    if match:
        return match.group(1)
    else:
        # If no '####', return the last number in the string
        numbers = re.findall(r'\d+', full_response)
        return numbers[-1] if numbers else ''

# Clean the full response and extract the final answer
df['cleaned_full_response'] = df['full_response'].apply(extract_final_answer)

# Convert all responses to strings for consistent comparison
df['model_response'] = df['model_response'].astype(str)
df['cleaned_full_response'] = df['cleaned_full_response'].astype(str)
df['correct_answer'] = df['correct_answer'].apply(lambda x: extract_final_answer(x)).astype(str)

# Compare model response to cleaned full response
df['model_matches_full'] = df['model_response'] == df['cleaned_full_response']

# Compare model response to correct answer
df['model_matches_correct'] = df['model_response'] == df['correct_answer']

# Calculate accuracies
accuracy_full = df['model_matches_full'].mean()
accuracy_correct = df['model_matches_correct'].mean()

print(f"Accuracy (model vs full response): {accuracy_full:.2%}")
print(f"Accuracy (model vs correct answer): {accuracy_correct:.2%}")

# Display some additional information
print("\nFirst few rows of the DataFrame:")
print(df[['question', 'model_response', 'cleaned_full_response', 'correct_answer', 'model_matches_full', 'model_matches_correct']].head())

print("\nValue counts of model_matches_full:")
print(df['model_matches_full'].value_counts(normalize=True))

print("\nValue counts of model_matches_correct:")
print(df['model_matches_correct'].value_counts(normalize=True))

Accuracy (model vs full response): 97.00%
Accuracy (model vs correct answer): 95.00%

First few rows of the DataFrame:
                                            question model_response  \
0  Janet’s ducks lay 16 eggs per day. She eats th...             18   
1  A robe takes 2 bolts of blue fiber and half th...              3   
2  Josh decides to try flipping a house.  He buys...          70000   
3  James decides to run 3 sprints 3 times a week....            540   
4  Every day, Wendi feeds each of her chickens th...             20   

  cleaned_full_response correct_answer  model_matches_full  \
0                    18             18                True   
1                     3              3                True   
2                 70000          70000                True   
3                   540            540                True   
4                    20             20                True   

   model_matches_correct  
0                   True  
1                   True  
